In [1]:
import random
import time

import pandas as pd
from generate_recipe_task import *
from experiment_classes import WrongIngTest

from together import Together

recipe_df = pd.read_csv("csv/inputs_outputs.csv")

num_steps = [len(row.iloc[5].split('\n')) for i, row in recipe_df.iterrows()]
num_ings = [len(row.iloc[3].split('\n')) for i, row in recipe_df.iterrows()]
recipe_df['num_steps'] = num_steps
recipe_df['num_ings'] = num_ings

long_recipes = recipe_df[recipe_df['num_steps'] >= 5]

long_recipes

,title,ingredients,directions,formatted_ingredients,formatted_ingredient_list,formatted_directions,num_steps,num_ings
0,One Hour Rolls,"[""1 c. milk"", ""2 Tbsp. sugar"", ""1 pkg. dry yea...","[""Put flour into a large mixing bowl."", ""Combi...","milk = Ingredient('m1lk', 'milk', 240, over='t...","ingredients = [milk, sugar, yeast, salt, oil, ...","t1 = Transformation('mix', 'Put flour into a l...",11,7
2,Fresh Strawberry Pie,"[""1 baked pie shell"", ""1 qt. cleaned strawberr...","[""Mix water, cornstarch, sugar and salt in sau...","pieshell = Ingredient('pshl', 'baked pie shell...","ingredients = [pieshell, strawberries, water, ...","t1 = Transformation('mix', 'Mix water, cornsta...",8,9
3,Summer Spaghetti,"[""1 lb. very thin spaghetti"", ""1/2 bottle McCo...","[""Prepare spaghetti per package."", ""Drain."", ""...","spaghetti = Ingredient('SPAG', 'very thin spag...","ingredients = [spaghetti, saladsupreme, zestyi...","t1 = Transformation('boil', 'Prepare spaghetti...",5,4
4,Rhubarb Coffee Cake,"[""1 1/2 c. sugar"", ""1/2 c. butter"", ""1 egg"", ""...","[""Cream sugar and butter."", ""Add egg and beat ...","sugar = Ingredient('sug1', 'sugar', 300, over=...","ingredients = [sugar, butter, egg, buttermilk1...","t1 = Transformation('mix', 'Cream sugar and bu...",6,12
9,Cheeseburger Potato Soup,"[""6 baking potatoes"", ""1 lb. of extra lean gro...","[""Wash potatoes; prick several times with a fo...","potato = Ingredient('POT1', 'baking potatoes',...","ingredients = [potato, beef, butter, milk, sal...","t1 = Transformation('mix', 'Wash potatoes; pri...",16,12
10,Mexican Cookie Rings,"[""1 1/2 c. sifted flour"", ""1/2 tsp. baking pow...","[""Sift flour, baking powder and salt together....","flour = Ingredient('flr1', 'sifted flour', 180...","ingredients = [flour, bpow, salt, butter, suga...","t1 = Transformation('mix', 'Sift flour, baking...",13,10
11,Chicken Divan,"[""1/4 c. margarine"", ""1/4 c. chopped onion (or...","[""Melt margarine in skillet; saute onions and ...","margarine = Ingredient('MARG', 'margarine', 57...","ingredients = [margarine, onion, celery, flour...","t1 = Transformation('fry', 'Melt margarine in ...",10,11
12,Spaghetti Sauce To Can,"[""1/2 bushel tomatoes"", ""1 c. oil"", ""1/4 c. mi...","[""Cook ground or chopped peppers and onions in...","tomatoes = Ingredient('tmt1', 'tomatoes', 1202...","ingredients = [tomatoes, oil, garlic, paste, p...","t1 = Transformation('fry', 'Cook ground or cho...",5,12
13,Taco Salad Chip Dip,"[""8 oz. Ortega taco sauce"", ""8 oz. sour cream""...","[""Mix taco sauce, sour cream and cream cheese....","sauce = Ingredient('sauc', 'Ortega taco sauce'...","ingredients = [sauce, sourcream, creamcheese, ...","t1 = Transformation('mix', 'Mix taco sauce, so...",7,12
15,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","peanut = Ingredient('PB01', 'peanut butter', 2...","ingredients = [peanut, graham, butter, sugar, ...","t1 = Transformation('mix', 'Combine first four...",5,6


In [2]:
num_recipes = 10

actor_specs = ["Alice", "She", 1, 0.5, 0]
transcipt_steps = 3
max_recipes = 4

In [3]:
selected_row = long_recipes.iloc[14]
print(f'Generating transcript for {selected_row.iloc[0]}')
generate_task_file_from_row(selected_row, actor_specs, transcipt_steps)
# time.sleep(1)
!python 'recipe_task/recipe_task.py'
# time.sleep(1)
output_text = open("recipe_task/output.txt", "r").read()
print(output_text)

Generating transcript for Taco-Filled Green Pepper
Alice fries the Ground Beef to make some brownedbeef.
She stirs the brownedbeef, taco seasoning, Kidney beans and salsa to make some meatmixture.
Continuing with the recipe, she boils the green peppers to make some parboiledpeppers.



In [4]:
aug_ings = ['10 oz garam masala', '5 cloves star anise']
test = WrongIngTest(output_text, transcipt_steps, selected_row, actor_specs, max_recipes=max_recipes, augmented_ingredients=aug_ings)

model_list = ["OpenAI/gpt-oss-20B",
              "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
              "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
              "moonshotai/Kimi-K2-Instruct-0905",
              "deepseek-ai/DeepSeek-R1",]

ing_acc = {}
for ing in test.unused_ingredients: ing_acc[ing] = []
overall_acc = []

for model in model_list:
    print(f'Testing {model}:')
    start = time.time()
    test.run_test(client=Together(api_key=''), model=model, n=num_recipes)
    test.print_results(match_exact=False)
    accuracies = test.get_results(match_exact=False)
    print(accuracies)

    print(f'Testing finished: ({'%.2f' % (time.time()-start)} seconds)')
    print('='*50)
    print()

    for i, ing in enumerate(test.unused_ingredients): ing_acc[ing].append(accuracies[i])
    overall_acc.append(sum(accuracies)/len(accuracies))

Testing OpenAI/gpt-oss-20B:
Running recipe with 'onion', spec level 0.
[**********]
Running recipe with 'Tomato', spec level 0.
[**********]
Running recipe with 'cheddar cheese', spec level 0.
[**********]
Expected: Recipe 2
Actual: ['Recipe 3', 'Recipe 3', 'Recipe 4', 'Recipe\xa03', 'Recipe 4', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 4']
Accuracy: 0.0
Expected: Recipe 3
Actual: ['Recipe 2', 'Recipe 2', 'Recipe 4', 'Recipe 2', 'Recipe 1', 'Recipe 2', 'Recipe 2', 'Recipe 4', 'Recipe 2', 'Recipe 2']
Accuracy: 0.0
Expected: Recipe 4
Actual: ['Recipe 2', 'Recipe 2', 'Recipe 1', 'Recipe 1', 'Recipe 2', 'Recipe 1', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2']
Accuracy: 0.0
[0.0, 0.0, 0.0]
Testing finished: (428.67 seconds)

Testing meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8:
Running recipe with 'onion', spec level 0.
[**********]
Running recipe with 'Tomato', spec level 0.
[**********]
Running recipe with 'cheddar cheese', spec level 0.
[**********]
Expected: Recipe

In [5]:
model = 'gpt-5'

print(f'Testing {model}:')
start = time.time()
test.run_test(n=num_recipes)
test.print_results(match_exact=False)
accuracies = test.get_results(match_exact=False)
print(accuracies)

print(f'Testing finished: ({'%.2f' % (time.time()-start)} seconds)')
print('='*50)
print()

model_list.append(model)
for i, ing in enumerate(test.unused_ingredients): ing_acc[ing].append(accuracies[i])
overall_acc.append(sum(accuracies)/len(accuracies))

Testing gpt-5:
Running recipe with 'onion', spec level 0.
[**********]
Running recipe with 'Tomato', spec level 0.
[**********]
Running recipe with 'cheddar cheese', spec level 0.
[**********]
Expected: Recipe 2
Actual: ['Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2', 'Recipe 2']
Accuracy: 1.0
Expected: Recipe 3
Actual: ['Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3', 'Recipe 3']
Accuracy: 1.0
Expected: Recipe 4
Actual: ['Recipe 4', 'Recipe 4', 'Recipe 4', 'Recipe 4', 'Recipe 4', 'Recipe 2', 'Recipe 4', 'Recipe 2', 'Recipe 4', 'Recipe 4']
Accuracy: 0.8
[1.0, 1.0, 0.8]
Testing finished: (648.12 seconds)



In [6]:
d = {'model': model_list}
for ing in test.unused_ingredients: d[ing] = ing_acc[ing]
d[f'overall accuracy (n={num_recipes * (max_recipes-1)})'] = overall_acc
df = pd.DataFrame(data=d)
df

,model,onion,Tomato,cheddar cheese,overall accuracy (n=30)
0,OpenAI/gpt-oss-20B,0.0,0.0,0.0,0.000000
1,meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8,0.0,0.0,0.0,0.000000
2,Qwen/Qwen3-235B-A22B-Instruct-2507-tput,0.0,0.0,0.0,0.000000
3,moonshotai/Kimi-K2-Instruct-0905,0.0,0.0,0.1,0.033333
4,deepseek-ai/DeepSeek-R1,0.9,0.7,0.7,0.766667
5,gpt-5,1.0,1.0,0.8,0.933333
